This notebook calculates the posteriors for GW190521 as a function of starting time for the FULL-SKY approach, and corresponding Bayes factor for the presence of the 330 mode

In [1]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=12"

import sys
sys.path.append('../../src/')

import numpy as onp
import jax
import jax.numpy as jnp

from myutils_corrected_2233 import Ntime, ACFs, set_detectors, set_detector_locations, cov_resp, initialize_det_locs
from myutils_corrected_2233 import ln_likelihood_full_jit
from scipy.linalg import toeplitz
import scipy.signal as sig

import lal
from gwpy.timeseries import TimeSeries
from other_utils import bandpass_ds, analysis_data, interp1d_jax, load_tables

jax.config.update("jax_enable_x64", True) 

/Users/kallol/Work/skyRing/examples/full-sky/../../src/myutils_corrected_2233.py:3: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


In [2]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from corner import corner
from chainconsumer import Chain, ChainConsumer, Truth, ChainConfig, PlotConfig

import numpyro
from numpyro.contrib.nested_sampling import NestedSampler
import numpyro.distributions as dist
numpyro.enable_x64()

/Users/kallol/miniconda3/envs/skyloc_ringdown/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


This uses data downloaded directly from gwpy.

Set the parameters to download strain.

In [3]:
tgps = 1242442967.445 + 0.006 
seglen = 8 
fs = 16384 
fmin = 8 
event_id = "GW190521" 
plot_checks = 0 
T = 0.2 
srate = 4096 

ra = 3.5   #right ascension
dec = 0.73 #declination


factor = 10
seed       = 31567
t0         = 0.0
fmax       = 1500

qnm1_path  = "../../data/l2/n1l2m2.dat"
qnm2_path  = "../../data/l3/n1l3m3.dat"

# Directory to save samples. 
tM_shifted_samples_save_dir = "../posterior_files/full-sky/GW190521-t-shift-posteriors/"
tM_shifted_220_samples_save_dir = "../posterior_files/full-sky/GW190521-220-t-shift-posteriors/"

Fetching initial strain data for H1 and L1

In [4]:
dH1 = TimeSeries.fetch_open_data('H1', tgps - seglen/2, tgps + seglen/2, sample_rate=fs)
dL1 = TimeSeries.fetch_open_data('L1', tgps - seglen/2, tgps + seglen/2, sample_rate=fs)

In [5]:
n_analyze = Ntime(srate, T)

# Calculate H1 and L1 times
delays = {}
tgps_ = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps_)
dt_ifo = delays.get('H1',
                    lal.TimeDelayFromEarthCenter(lal.CachedDetectors[lal.LALDetectorIndexLHODIFF].location, ra, dec, tgps))
tH1 = tgps + dt_ifo

delays = {}
tgps_ = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps_)
dt_ifo = delays.get('L1',
                    lal.TimeDelayFromEarthCenter(lal.CachedDetectors[lal.LALDetectorIndexLLODIFF].location, ra, dec, tgps))
tL1 = tgps + dt_ifo

Calculate PSDs directly from the data.

In [6]:
nperseg = int(seglen * (1/(dH1.times.value[1] - dH1.times.value[0])))
noverlap = nperseg // 2

psdH = sig.welch(
    dH1.value,
    fs=fs,
    window="hann",
    nperseg=4096*4,
    noverlap=4096*2,
    detrend=False,
    return_onesided=True,
    scaling="density",    
)

psdL = sig.welch(
    dL1.value,
    fs=fs,
    window="hann",
    nperseg=4096*4,
    noverlap=4096*2,
    detrend=False,
    return_onesided=True,
    scaling="density",   
)

Set priors for the PE

In [7]:
# Prior limits
limits = [
    [30.0, 500.0],
    [0., 0.99],
    [0., 4.0],
    [0., 6.283185307179586],
    [0., 4.0],
    [0., 6.283185307179586],
    [-1., 1.],
    [0, 6.28],
    [-1., 1.],
    [0, 3.14]]

low  = onp.asarray([pair[0] for pair in limits], dtype=onp.float64)
high = onp.asarray([pair[1] for pair in limits], dtype=onp.float64)

Interpolating PSDs over freq range, setting up cov (using Cholesky decomposition) for TD likelihood

In [8]:
psdH = interp1d_jax(jnp.asarray(psdH[0],dtype=jnp.float64), jnp.asarray(psdH[1],dtype=jnp.float64))
psdL = interp1d_jax(jnp.asarray(psdL[0],dtype=jnp.float64), jnp.asarray(psdL[1],dtype=jnp.float64))

omega_22_r, omega_22_i, omega_33_r, omega_33_i = load_tables(qnm1_path, qnm2_path)
tgps = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps)

L_H, L_L, resp_H, resp_L = cov_resp(srate=srate, T=T, psdH=psdH, psdL=psdL, factor=factor)
set_detectors(resp_H, resp_L)

rH, rL = initialize_det_locs()
set_detector_locations(rH, rL)

The starting time is changed in increments of $t_M$, where $M=325M_\odot$. $M$ is selected from the posterior peak at $t_{\rm ref}$

In [9]:
MTSUN = lal.MTSUN_SI
tM = MTSUN * 325 # Remnant mass peak

Setting up the data with different starting times, in increments of $t_M$

For FULL-SKY analysis, H1 and L1 share a common starting time. This is selected as the H1 time.

In [10]:
num_shifts = 12

t_shifts = []
H1_data_t_shifted = []
L1_data_t_shifted = []

for jj in range(num_shifts):

    dH1_cond = bandpass_ds(dH1, t0=tH1 + (jj)*1*tM, ds=int(fs/srate), trim=0.25, f_min=fmin)
    dL1_cond = bandpass_ds(dL1, t0=tH1 + (jj)*1*tM, ds=int(fs/srate), trim=0.25, f_min=fmin)

    dH1_analysis_data = analysis_data(dH1_cond[1], dH1_cond[0], tH1 + (jj)*1*tM, n_analyze)
    dL1_analysis_data = analysis_data(dL1_cond[1], dL1_cond[0], tH1 + (jj)*1*tM, n_analyze)

    t_shifts.append((jj)*1*tM)
    H1_data_t_shifted.append(dH1_analysis_data[1])
    L1_data_t_shifted.append(dL1_analysis_data[1])

Save the time increments (in sec)

In [12]:
onp.savetxt(tM_shifted_samples_save_dir + 't_shifts.txt', t_shifts)

Set up the numpyro model. Computes likelihood as a function of $\theta$

In [13]:
low  = jnp.asarray(low,  dtype=jnp.float64)
high = jnp.asarray(high, dtype=jnp.float64)

In [ ]:
ref_det = 'H1'
logZs = []
for i in range(num_shifts):
    h_H_t = H1_data_t_shifted[i]
    h_L_t = L1_data_t_shifted[i]

    def make_loglik_fn(dataH, dataL, gmst, L_H, L_L,
                    omega_22_r, omega_22_i, omega_33_r, omega_33_i,
                    T, srate, t0):
        def _loglik(theta):
            return ln_likelihood_full_jit(
                dataH=dataH, dataL=dataL, params=theta,
                gmst=gmst, L_H=L_H, L_L=L_L,
                omega_22_r=omega_22_r, omega_22_i=omega_22_i,
                omega_33_r=omega_33_r, omega_33_i=omega_33_i,
                T=T, srate=srate, t0=t0, ref_de=ref_det
            )
        return _loglik

    loglik_fn = make_loglik_fn(
        dataH=h_H_t, dataL=h_L_t,
        gmst=gmst, L_H=L_H, L_L=L_L,
        omega_22_r=omega_22_r, omega_22_i=omega_22_i,
        omega_33_r=omega_33_r, omega_33_i=omega_33_i,
        T=T, srate=srate, t0=t0
    )

    def model():
        theta = numpyro.sample("theta", dist.Uniform(low, high).to_event(1))
        numpyro.factor("loglike", loglik_fn(theta))

    rng_key = jax.random.PRNGKey(int(seed) ^ 0xABCDEF)
    rng_run, rng_post = jax.random.split(rng_key)
    ns = NestedSampler(
        model,
        constructor_kwargs=dict(
            num_live_points=10000,    
            max_samples=500_000,
            verbose=False,
        ),
        termination_kwargs=dict(
            dlogZ=0.01,
        ),
    )
    ns.run(rng_run)
    ns.print_summary()
    posterior = ns.get_samples(rng_post, num_samples=100_000)

    onp.save(tM_shifted_samples_save_dir + 'posterior.' + event_id + '.' + str(i) + '.npy', onp.asarray(posterior['theta']))
    logZs.append(ns._results.log_Z_mean)
    onp.savetxt(tM_shifted_samples_save_dir + 'logZs.txt', onp.asarray(logZs))


Now we perform the same analysis with the data, but assuming only the 220 mode in the waveform model

In [ ]:
# No 330 mode
limits = [
    [30.0, 500.0],
    [0., 0.99],
    [0., 4.0],
    [0., 6.283185307179586],
    [-1., 1.],
    [0, 6.28],
    [-1., 1.],
    [0, 3.14]]

low  = onp.asarray([pair[0] for pair in limits], dtype=onp.float64)
high = onp.asarray([pair[1] for pair in limits], dtype=onp.float64)

In [17]:
low  = jnp.asarray(low,  dtype=jnp.float64)
high = jnp.asarray(high, dtype=jnp.float64)

Set up the numpyro model. Computes likelihood as a function of $\theta$

In [ ]:
logZs_220 = []
for i in range(num_shifts):
    h_H_t = H1_data_t_shifted[i]
    h_L_t = L1_data_t_shifted[i]

    def make_loglik_fn(dataH, dataL, gmst, L_H, L_L,
                    omega_22_r, omega_22_i, omega_33_r, omega_33_i,
                    T, srate, t0):
        def _loglik(theta):
            return ln_likelihood_full_jit(
                dataH=dataH, dataL=dataL, params=theta,
                gmst=gmst, L_H=L_H, L_L=L_L,
                omega_22_r=omega_22_r, omega_22_i=omega_22_i,
                omega_33_r=omega_33_r, omega_33_i=omega_33_i,
                T=T, srate=srate, t0=t0, ref_det=ref_det
            )
        return _loglik

    loglik_fn = make_loglik_fn(
        dataH=h_H_t, dataL=h_L_t,
        gmst=gmst, L_H=L_H, L_L=L_L,
        omega_22_r=omega_22_r, omega_22_i=omega_22_i,
        omega_33_r=omega_33_r, omega_33_i=omega_33_i,
        T=T, srate=srate, t0=t0
    )

    def model_220():
        theta = numpyro.sample("theta", dist.Uniform(low, high).to_event(1))
        amp_phase_330 = jnp.array([0.0, 0.0])
        theta_ = jnp.concatenate(
                            [
                                theta[:4],
                                amp_phase_330,
                                theta[4:],
                            ],
                        )

        numpyro.factor("loglike", loglik_fn(theta_))

    rng_key = jax.random.PRNGKey(int(seed) ^ 0xABCDEF)
    rng_run, rng_post = jax.random.split(rng_key)
    ns = NestedSampler(
        model_220,
        constructor_kwargs=dict(
            num_live_points=10000,    
            max_samples=500_000,
            verbose=False,
        ),
        termination_kwargs=dict(
            dlogZ=0.01,
        ),
    )
    ns.run(rng_run)
    ns.print_summary()
    posterior = ns.get_samples(rng_post, num_samples=100_000)

    onp.save(tM_shifted_220_samples_save_dir + 'posterior.' + event_id + '.' + str(i) + '.npy', onp.asarray(posterior['theta']))
    logZs_220.append(ns._results.log_Z_mean)
    onp.savetxt(tM_shifted_220_samples_save_dir + 'logZs_220.txt', onp.asarray(logZs_220))


To plot the results, check the plots.ipynb file in the examples directory